# Phase 6 — Risk Assessment Agent

> Run AFTER Phase 6 (restart the kernel first). Cells 1–4 are LLM-free; cell 5 calls Gemini.

**Data fact:** client profiles are **synthetic** (the source file has only holdings). Two deliberate mismatches are seeded: CLT-009 (conservative profile, speculative tech book) and CLT-003 (aggressive profile, index+cash book).


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env'
print('env OK')

env OK


## 1. The synthetic profiles (documented assumption)

In [2]:
from app.data.repositories import profile_repo
for cid in ['CLT-001', 'CLT-003', 'CLT-009']:
    p = profile_repo.get(cid)
    print(f'{cid}: {p.name:20} {p.risk_tolerance:12} score={p.risk_score} '
          f'horizon={p.time_horizon_years}y income_needs={p.income_needs}')

CLT-001: Margaret Whitfield   conservative score=3 horizon=8y income_needs=True
CLT-003: Priya Raghavan       aggressive   score=9 horizon=25y income_needs=False
CLT-009: Sofia Marchetti      conservative score=2 horizon=5y income_needs=True


## 2. Risk metrics (deterministic math on real prices — no LLM)

In [3]:
from app.tools.risk_tools import portfolio_volatility, portfolio_beta, value_at_risk
print('CLT-009 volatility:', portfolio_volatility('CLT-009'))
print('CLT-001 volatility:', portfolio_volatility('CLT-001'))
print('CLT-009 beta      :', portfolio_beta('CLT-009'))

{"rows": 84, "clients": 10, "event": "portfolios_loaded", "level": "info", "timestamp": "2026-07-12T20:50:47.369209Z"}
{"method": "get_price_history", "adapter": "finnhub", "error": "[finnhub] price history requires a paid plan", "event": "chain_adapter_failed", "level": "warning", "timestamp": "2026-07-12T20:50:47.372201Z"}
{"method": "get_price_history", "adapter": "alpha_vantage", "error": "[alpha_vantage] period '1y' unsupported on free tier", "event": "chain_adapter_failed", "level": "warning", "timestamp": "2026-07-12T20:50:47.372703Z"}
{"method": "get_price_history", "source": "yfinance", "event": "chain_served", "level": "info", "timestamp": "2026-07-12T20:50:47.869415Z"}
{"method": "get_price_history", "adapter": "finnhub", "error": "[finnhub] price history requires a paid plan", "event": "chain_adapter_failed", "level": "warning", "timestamp": "2026-07-12T20:50:47.871739Z"}
{"method": "get_price_history", "adapter": "alpha_vantage", "error": "[alpha_vantage] period '1y' unsup

## 3. VaR — both Strategy implementations side by side

In [4]:
for m in ('historical', 'parametric'):
    r = value_at_risk('CLT-009', method=m)
    print(f"{m:11} VaR(95%) 1d: {r['var_1d_pct']}%  (~${r['var_1d_dollars']:,.0f})")

{"fn": "_history", "age_s": 22.7, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:10.561620Z"}
{"fn": "_history", "age_s": 22.6, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:10.564538Z"}
{"fn": "_history", "age_s": 22.5, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:10.567044Z"}
{"fn": "_history", "age_s": 22.3, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:10.569111Z"}
{"fn": "_history", "age_s": 22.2, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:10.571786Z"}
{"fn": "_history", "age_s": 22.1, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:10.574143Z"}
{"fn": "_history", "age_s": 22.0, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:10.579989Z"}
{"fn": "_history", "age_s": 21.9, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:10.582211Z"}
{"fn": "_history", "age_s": 21.7, "event": "cache_hit", "level":

## 4. Concentration + the tolerance mismatch (the demo centerpiece)

In [5]:
from app.tools.risk_tools import concentration_metrics, risk_tolerance_check
c = concentration_metrics('CLT-009')
print('asset split:', c['asset_class_split_pct'], '| largest sector:', c['largest_sector'])
for cid in ['CLT-009', 'CLT-001', 'CLT-003']:
    r = risk_tolerance_check(cid)
    print(f"{cid}: mismatch={r['mismatch']} — {r['rationale']}")

{"fn": "_quote", "age_s": 26.5, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:17.526144Z"}
{"fn": "_quote", "age_s": 25.4, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:17.527051Z"}
{"fn": "_quote", "age_s": 24.9, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:17.527907Z"}
{"fn": "_quote", "age_s": 23.8, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:17.528246Z"}
{"fn": "_quote", "age_s": 22.6, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:17.528821Z"}
{"fn": "_quote", "age_s": 21.6, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:17.529221Z"}
{"fn": "_quote", "age_s": 21.0, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:17.529721Z"}
{"fn": "_quote", "age_s": 19.8, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T20:51:17.530264Z"}
{"fn": "_quote", "age_s": 19.4, "event": "cache_hit", "level": "info", "timestam

## 5. The risk agent narrates it (Gemini — split first, numbers contextualized)

In [6]:
from app.agents.risk import RiskAssessmentAgent
print(RiskAssessmentAgent().answer(client_id='CLT-009',
    query='What is my risk exposure and is it aligned with my risk tolerance?'))

{"query": "What is my risk exposure and is it aligned with my risk tolerance?", "event": "agent_start", "agent": "risk", "client_id": "CLT-009", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T20:51:31.073524Z"}


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"fn": "_quote", "age_s": 48.6, "event": "cache_hit", "agent": "risk", "client_id": "CLT-009", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T20:51:39.567896Z"}
{"fn": "_history", "age_s": 51.7, "event": "cache_hit", "agent": "risk", "client_id": "CLT-009", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T20:51:39.573108Z"}
{"fn": "_quote", "age_s": 47.4, "event": "cache_hit", "agent": "risk", "client_id": "CLT-009", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T20:51:39.574223Z"}
{"fn": "_history", "age_s": 51.6, "event": "cache_hit", "agent": "risk", "client_id": "CLT-009", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T20:51:39.578792Z"}
{"fn": "_quote", "age_s": 48.6, "event": "cache_hit", "agent": "risk", "client_id": "CLT-009", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T20:51:39.580711Z"}
{"fn": "_history", "age_s": 51.7, "event": "cache_hit", "agent": "risk", "client_id": "CLT-009", "ses

HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"seconds": 17.44, "new_messages": 7, "tool_calls": 5, "event": "agent_done", "agent": "risk", "client_id": "CLT-009", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T20:51:48.518439Z"}
As the Risk Assessment officer for XZY Capital, I have completed the review of your portfolio (CLT-009).

### Asset-Class Composition
Your portfolio is currently allocated as follows:
*   **Equity:** 89.34%
*   **Cash:** 10.66%

This heavy concentration in equities serves as the primary driver for the risk metrics detailed below.

### Risk Exposure Metrics
*   **Volatility:** The portfolio has an annualized volatility of 33.15%. This indicates a high degree of fluctuation in your portfolio's value over the past 250 trading days.
*   **Beta:** The equity sleeve of your portfolio has a beta of 1.58 relative to the S&P 500 (SPY). This means that for every 1% move in the broader market, your equity holdings tend to move approximately 58% more, indicating significant sensitivity to market sw

## ✅ Acceptance check

In [7]:
assert risk_tolerance_check('CLT-009')['mismatch'] is True
assert risk_tolerance_check('CLT-001')['mismatch'] is False
assert risk_tolerance_check('CLT-003')['mismatch'] is True   # conservative-direction mismatch
print('Phase 6 acceptance: PASS')

{"method": "get_quote", "source": "finnhub", "event": "chain_served", "agent": "risk", "client_id": "CLT-009", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T20:51:59.603743Z"}
{"method": "get_quote", "source": "finnhub", "event": "chain_served", "agent": "risk", "client_id": "CLT-009", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T20:52:00.115943Z"}
{"method": "get_quote", "source": "finnhub", "event": "chain_served", "agent": "risk", "client_id": "CLT-009", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T20:52:00.629377Z"}
{"method": "get_quote", "source": "finnhub", "event": "chain_served", "agent": "risk", "client_id": "CLT-009", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T20:52:01.097789Z"}
{"method": "get_quote", "source": "finnhub", "event": "chain_served", "agent": "risk", "client_id": "CLT-009", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T20:52:02.368627Z"}
{"method": "get_quote", "